# Lesson 8: Layer normalization

Karpathy introduces a small `LayerNorm1d` example after attention scaling. This lesson makes its normalization axis explicit: each token vector is normalized across its channels, independently of the other tokens in the batch.

The matching [C lesson](../c/src/layer_norm_main.c) and [Java lesson](../java/src/main/java/learning/karpathy/gpt1/LayerNormLesson.java) use the same tiny input. See the [lesson map](../README.md#run-lessons-by-step) for running selected steps.

References: [PyTorch `nn.LayerNorm`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html) and the original [Layer Normalization paper](https://arxiv.org/abs/1607.06450).

## Step 1: Choose the dimension being normalized

Our input has shape `(batch, time, channels)`. At each batch and time position, LayerNorm computes a mean and variance across that token's channel values. It does not use other examples or other time positions to calculate those statistics. This makes it different from BatchNorm, whose statistics are computed across examples during training.

The source tutorial calls its toy class `LayerNorm1d` and applies it to a 2D `(batch, features)` tensor. Here we keep the time dimension visible and normalize the last dimension, as `nn.LayerNorm(channels)` does in a Transformer.

In [ ]:
import torch
from torch import nn

torch.set_printoptions(precision=4, sci_mode=False)

# Two sequences, two token positions each, four channels per token.
x = torch.tensor([
    [[1.0, 2.0, 3.0, 4.0], [2.0, 4.0, 6.0, 8.0]],
    [[10.0, 11.0, 12.0, 13.0], [3.0, 1.0, 4.0, 2.0]],
])
batch_size, time_steps, channels = x.shape
print(f"shape: {tuple(x.shape)} = (batch={batch_size}, time={time_steps}, channels={channels})")
print("Input token vectors:")
print(x)

# Statistics are local to each token vector (the last dimension).
token_means = x.mean(dim=-1)
token_variances = x.var(dim=-1, correction=0)
print("Per-token means:")
print(token_means)
print("Per-token population variances:")
print(token_variances)

## Step 2: Center and rescale each token vector

For one token vector `x`, LayerNorm computes `x_hat = (x - mean(x)) / sqrt(var(x) + eps)`. We use the population variance (`correction=0`), which is what PyTorch's `nn.LayerNorm` uses. The original toy cell calls `x.var(...)` with PyTorch's default sample-variance correction, so its values differ slightly from `nn.LayerNorm`. The `eps` term keeps division safe when the variance is zero.

After this step, each non-constant vector has mean near zero and population standard deviation near one. A channel's average across different tokens is not required to be zero; the guarantee is per token.

In [ ]:
eps = 1e-5
x_hat = (x - token_means[..., None]) / torch.sqrt(token_variances[..., None] + eps)

print("Normalized token vectors:")
print(x_hat)
print("Means per token after normalization:")
print(x_hat.mean(dim=-1))
print("Population standard deviations per token:")
print(x_hat.std(dim=-1, correction=0))
print("Mean of channel 0 across all tokens (not constrained by LayerNorm):")
print(x_hat[..., 0].mean())

## Step 3: Learn a scale and a shift per channel

A Transformer LayerNorm has trainable `gamma` (scale) and `beta` (shift) values for each channel: `y = gamma * x_hat + beta`. These let later layers choose a useful representation instead of being forced to keep exactly zero mean and unit variance. The source snippet's plain tensors illustrate the values; to train them inside an `nn.Module`, they must be registered as `nn.Parameter`. PyTorch's `nn.LayerNorm` handles that automatically; here we set example values so the calculation is visible.

The original tutorial's hand-written class illustrates this formula. In this guided version, `nn.LayerNorm` is also run with the same parameters to verify the library implementation.

In [ ]:
gamma = torch.tensor([1.0, 0.5, 2.0, -1.0])
beta = torch.tensor([0.0, 1.0, -1.0, 0.5])
manual_output = gamma * x_hat + beta

layer_norm = nn.LayerNorm(channels, eps=eps)
with torch.no_grad():
    layer_norm.weight.copy_(gamma)
    layer_norm.bias.copy_(beta)
library_output = layer_norm(x)

print("gamma (per-channel scale):", gamma)
print("beta (per-channel shift):", beta)
print("Output after scale and shift:")
print(manual_output)
print("Matches nn.LayerNorm:", torch.allclose(manual_output, library_output, atol=1e-6))
print()
print("In a neural network, backpropagation updates gamma and beta along with the other model parameters.")